# Extended Sentinel-2 Image Collection

Data and model artifacts are intentionally external to this repository. Configure their locations in `.env` before running the notebook.


In [ ]:
# Reproducible project paths (launch Jupyter from the repository root)
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "README.md").is_file():
    PROJECT_ROOT = next(
        (parent for parent in PROJECT_ROOT.parents if (parent / "README.md").is_file()),
        PROJECT_ROOT,
    )

load_dotenv(PROJECT_ROOT / ".env")
DATA_ROOT = Path(os.getenv("SOIL_DATA_DIR", PROJECT_ROOT / "data")).expanduser()
MODEL_ROOT = Path(os.getenv("SOIL_MODEL_DIR", PROJECT_ROOT / "models")).expanduser()
OUTPUT_ROOT = Path(os.getenv("SOIL_OUTPUT_DIR", PROJECT_ROOT / "outputs")).expanduser()

if not DATA_ROOT.is_absolute():
    DATA_ROOT = PROJECT_ROOT / DATA_ROOT
if not MODEL_ROOT.is_absolute():
    MODEL_ROOT = PROJECT_ROOT / MODEL_ROOT
if not OUTPUT_ROOT.is_absolute():
    OUTPUT_ROOT = PROJECT_ROOT / OUTPUT_ROOT

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / ".cache").mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / ".cache").mkdir(parents=True, exist_ok=True)


In [ ]:
SH_CLIENT_ID = os.getenv("SH_CLIENT_ID", "").strip()
SH_CLIENT_SECRET = os.getenv("SH_CLIENT_SECRET", "").strip()
if not SH_CLIENT_ID or not SH_CLIENT_SECRET:
    raise RuntimeError("Set SH_CLIENT_ID and SH_CLIENT_SECRET in .env before collecting imagery.")


# 2024 0.1

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

from sentinelhub import (
    SHConfig, SentinelHubRequest, SentinelHubCatalog, DataCollection, MimeType,
    BBox, CRS, bbox_to_dimensions
)
import os
import time
import shutil

import pandas as pd
import os
from datetime import datetime

from sentinelhub import BBox, CRS

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.01
resolution = 10
max_cloud = 30
year = "2024"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2024_10_1"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2024_10_1/tile_0/2024-01-15.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2024 0.2

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.02
resolution = 10
max_cloud = 30
year = "2024"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2024_10_2"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2024_10_2/tile_0/2024-01-15.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2024 0.3

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.03
resolution = 10
max_cloud = 30
year = "2024"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2024_10_3"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2024_10_3/tile_0/2024-01-15.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2023 0.1

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.01
resolution = 10
max_cloud = 30
year = "2023"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2023_10_1"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2023_10_1/tile_0/2023-01-20.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2023 0.2

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.02
resolution = 10
max_cloud = 30
year = "2023"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2023_10_2"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2023_10_2/tile_0/2023-01-20.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2023_10_2/tile_0/2023-08-08.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2023 0.3

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.03
resolution = 10
max_cloud = 30
year = "2023"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2023_10_3"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2023_10_3/tile_0/2023-08-08.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2022 0.2

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.02
resolution = 10
max_cloud = 30
year = "2022"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2022_10_2"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2022_10_2/tile_0/2022-01-25.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2022 0.3

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.03
resolution = 10
max_cloud = 30
year = "2022"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2022_10_3"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2022_10_3/tile_0/2022-08-08.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2021 0.3

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.03
resolution = 10
max_cloud = 30
year = "2021"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2021_10_3"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2021_10_3/tile_0/2021-08-08.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2020 0.3

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.03
resolution = 10
max_cloud = 30
year = "2020"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2020_10_3"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2020_10_3/tile_0/2020-08-08.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2019 0.3

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.03
resolution = 10
max_cloud = 30
year = "2019"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2019_10_3"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2019_10_3/tile_0/2019-08-04.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2018 0.3

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.03
resolution = 10
max_cloud = 30
year = "2018"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2018_10_3"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2018_10_3/tile_0/2018-08-04.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()

# 2017 0.3

In [ ]:
# Sentinel Hub credentials
config = SHConfig()
config.sh_client_id = SH_CLIENT_ID
config.sh_client_secret = SH_CLIENT_SECRET

# Parameters
window_size = 0.03
resolution = 10
max_cloud = 30
year = "2017"
output_dir = str(DATA_ROOT / Path("sentinel_downloads_2017_10_3"))

# Area bounds
start_lat = 32.19625
start_lon = 35.60446
end_lat = 32.161213
end_lon = 35.615343

# Evalscript to request B8A and B11
evalscript = """
//VERSION=3
function setup() {
  return {
    input: ["B8A", "B11"],
    output: {
      bands: 2,
      sampleType: "FLOAT32",
      noDataValue: -9999
    }
  };
}

function evaluatePixel(sample) {
  return [sample.B8A, sample.B11];
}
"""

# Catalog setup
catalog = SentinelHubCatalog(config=config)

# Updated function to fetch available dates using CQL2
def get_available_dates(bbox_coords, time_interval):
    try:
        bbox = BBox(bbox=bbox_coords, crs=CRS.WGS84)
        search_iterator = catalog.search(
            collection=DataCollection.SENTINEL2_L2A,
            bbox=bbox,
            time=time_interval,
            filter=f"eo:cloud_cover < {max_cloud}",
            fields={"include": ["properties.datetime"], "exclude": []}
        )
        return sorted({item["properties"]["datetime"][:10] for item in search_iterator})
    except Exception as e:
        print(f"Error fetching dates for bbox {bbox_coords}: {e}")
        return []

# Generate 1×1 km tile bounding boxes
tile_bboxes = []
lat = start_lat
while lat >= end_lat:
    lon = start_lon
    while lon <= end_lon:
        tile_bboxes.append([lon, lat, lon + window_size, lat + window_size])
        lon += window_size
    lat -= window_size

# Process tiles
for tile_id, tile_coords in enumerate(tile_bboxes):
    tile_folder = os.path.join(output_dir, f"tile_{tile_id}")
    os.makedirs(tile_folder, exist_ok=True)

    print(f"\nProcessing tile {tile_id} - BBOX: {tile_coords}")
    dates = get_available_dates(tile_coords, (f"{year}-01-01", f"{year}-12-31"))
    print(f"Found {len(dates)} dates for tile {tile_id}")

    bbox_obj = BBox(bbox=tile_coords, crs=CRS.WGS84)
    size = bbox_to_dimensions(bbox_obj, resolution=resolution)

    for date in dates:
        output_path = os.path.join(tile_folder, f"{date}.tiff")
        if os.path.exists(output_path):
            print(f"Already downloaded: {output_path}")
            continue

        try:
            request = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=evalscript,
                input_data=[
                    {
                        "type": "sentinel-2-l2a",
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date}T00:00:00Z",
                                "to": f"{date}T23:59:59Z"
                            }
                        }
                    }
                ],
                responses=[
                    {
                        "identifier": "default",
                        "format": {
                            "type": MimeType.TIFF.get_string()
                        }
                    }
                ],
                bbox=bbox_obj,
                size=size,
                config=config
            )

            results = request.get_data(save_data=True)

            # Locate and rename the file
            # It will be in: tile_folder/hash_folder/response.tiff
            hash_folder = max(
                [os.path.join(tile_folder, d) for d in os.listdir(tile_folder)],
                key=os.path.getmtime
            )

            response_path = os.path.join(hash_folder, "response.tiff")

            if os.path.exists(response_path):
                shutil.move(response_path, output_path)
                shutil.rmtree(hash_folder)
                print(f" Saved: {output_path}")
            else:
                print(f" Warning: response.tiff not found for {date}")

            time.sleep(1)

        except Exception as e:
            print(f" Error on {date} for tile {tile_id}: {e}")

In [ ]:
# Load the .tiff file
tiff_path = str(DATA_ROOT / Path("sentinel_downloads_2017_10_3/tile_0/2017-08-04.tiff"))

with rasterio.open(tiff_path) as src:
    image = src.read()  # shape: (2, H, W)
    profile = src.profile

# Split bands
B8A = image[0, :, :]  # Near-Infrared
B11 = image[1, :, :]  # SWIR

# Print shapes and basic stats
print("B8A shape:", B8A.shape, "min/max:", B8A.min(), B8A.max())
print("B11 shape:", B11.shape, "min/max:", B11.min(), B11.max())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(B8A, cmap='gray')
ax[0].set_title("B8A (NIR)")

ax[1].imshow(B11, cmap='gray')
ax[1].set_title("B11 (SWIR)")

plt.tight_layout()
plt.show()

In [ ]:
# Avoid division by zero
denom = B8A + B11
msi = np.where(denom == 0, 0, (B11 - B8A) / denom)

plt.imshow(msi, cmap='viridis')
plt.colorbar(label='MSI')
plt.title('Moisture Stress Index')
plt.show()